In [0]:
%python
# DBTITLE 1,Load model and make predictions
import mlflow
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array

# 1. Define the model path in Unity Catalog
model_uri = "models:/ai2605.ai.petstreaming_sales_model/1"
input_df=spark.table("ai2605.ai.petstreaming_sales")

# 2. Load the model from registry
loaded_model = mlflow.spark.load_model(model_uri,dfs_tmpdir="/Volumes/ai2605/ai/ml_temp")

# 3. Apply the model to the input DataFrame
predictions = loaded_model.transform(input_df)

#4. Display the results (with prediction probabilities)
#display(predictions.select("category", "units_sold","price","prediction"))
display(
    predictions.select(
        "category", "units_sold", "price", "prediction",
        vector_to_array("probability").alias("prob_array")
    )
)


In [0]:
%python
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# 1. Define the model path in Unity Catalog
model_uri = "models:/ai2605.ai.petstreaming_sales_model/1"

spark_df = spark.table("ai2605.ai.petstreaming_sales")
df_with_label = spark_df.withColumn("label", col("is_subscription").cast("double"))
loaded_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/ai2605/ai/ml_temp" # Point to your UC Volume
)

# 3. Use the model to make predictions
    # You can use the loaded_model just like the original pipeline object
predictions = loaded_model.transform(df_with_label)

# Updated selection to include label and probability
display(
    predictions.select(
        "category", 
        "units_sold", 
        "price", 
        "label",           # The actual historical value
        "prediction",      # The model's final decision
        "probability"      # The model's confidence (Vector)
    )
)



confusion_matrix = predictions.crosstab("label", "prediction")

display(confusion_matrix)
from pyspark.sql.functions import col

# Since your columns are named '0.0' and '1.0', we must access them as strings
def get_val(actual, pred):
    # actual is 0.0 or 1.0 (float)
    # pred is 0.0 or 1.0 (float)
    # Convert pred to string to match the column name e.g., '0.0'
    col_name = str(float(pred)) 
    return confusion_matrix.filter(col("label_prediction") == actual).select(col("`" + col_name + "`")).collect()[0][0]

# Retrieve the four components
tn = get_val(0.0, 0.0) # Actual 0.0, Pred 0.0
fp = get_val(0.0, 1.0) # Actual 0.0, Pred 1.0
fn = get_val(1.0, 0.0) # Actual 1.0, Pred 0.0
tp = get_val(1.0, 1.0) # Actual 1.0, Pred 1.0

# Calculate metrics
total = tn + fp + fn + tp
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Log to MLflow
with mlflow.start_run(run_name="ml_0625_01"):
    mlflow.log_metrics({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })
    print(f"Metrics logged: Acc={accuracy:.4f}, Prec={precision:.4f}, Rec={recall:.4f}, F1={f1:.4f}")

In [0]:
%python
import mlflow
model_info = mlflow.models.get_model_info("models:/ai2605.ai.petstreaming_sales_model/2")
print(model_info.signature)

In [0]:
%python
import mlflow
import mlflow.spark
import pandas as pd
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from mlflow.models.signature import infer_signature

# 1. READ DATA (This is where the data enters the pipeline)
# Ensure the table name matches your environment
spark_df = spark.table("ai2605.ai.petstreaming_sales")

# 2. PREPARE LABEL
from pyspark.sql.functions import col
df_prepared = spark_df.withColumn("label", col("is_subscription").cast("double"))


# 3. PIPELINE DEFINITION
indexer = StringIndexer(inputCol="category", outputCol="category_idx")
encoder = OneHotEncoder(inputCol="category_idx", outputCol="category_vec")
assembler = VectorAssembler(inputCols=["units_sold", "price", "category_vec"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Custom transformer to prune unwanted Spark vectors before they hit the Serving Layer
#class PredictionCleaner(Transformer, DefaultParamsReadable, DefaultParamsWritable):
#    def _transform(self, dataset):
#        return dataset.select("prediction")

# 4. TRAIN
#pipeline = Pipeline(stages=[indexer, encoder, assembler, lr, PredictionCleaner()])
pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(df_prepared)

# 5. DEFINE CONTRACT (Signature)
# We infer it from a clean dataframe that matches the inputs expected by the API
input_example = df_prepared.select("units_sold", "price", "category").limit(1).toPandas()
output_example = pd.DataFrame({"prediction": [0.0]})
signature = infer_signature(input_example, output_example)

# 6. LOG AND REGISTER
model_name = "ai2605.ai.petstreaming_sales_model"
volume_path = "/Volumes/ai2605/ai/ml_temp"
with mlflow.start_run(run_name="production_v1"):
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        signature=signature,
        registered_model_name=model_name,
        dfs_tmpdir=volume_path
    )

print(f"Model successfully logged and registered as: {model_name}")
     